# AutoSolve Standalone ML Training Pipeline 🚀

Welcome! This notebook is a **completely self-contained training suite** for the AutoSolve Blender camera tracking addon. It contains all feature extraction, PyTorch training loops, and ONNX/NumPy exporters inside the cells below.

---

## 📖 Easy 3-Step Google Colab Flow:

### 1. Upload Your Videos
* In your **Google Drive**, create a folder named `AutoSolve_ML_Data` and a subfolder named `clips` inside it (i.e. `AutoSolve_ML_Data/clips`).
* Upload all your video clips (.mp4, .mkv, .mov, etc.) into the `clips` folder.

### 2. Run All Cells
* In the menu above, click **Runtime → Run all** (or run each cell sequentially from top to bottom by pressing `Shift + Enter`).
* When prompted in the second cell, authorize Google Drive access.

### 3. Copy Generated Models to Blender Addon
Once training completes, download the exported files from the Colab file browser and copy them into your local Blender addon folders:

* **ONNX & NumPy Models:** Copy `settings_model.onnx`, `track_predictor.onnx`, `patch_rigidity.onnx`, and `track_predictor.json` to:
  📂 `autosolve/tracker/models/`

* **Presets:** Copy `region_weights.json` and `defaults.json` to:
  📂 `autosolve/tracker/presets/`

---


## 📂 Google Drive Persistent Storage (Recommended for Colab)
Link directories directly to your Google Drive to persist video clips and trained checkpoints.

In [ ]:
#@title Configure Google Drive Integration
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
DRIVE_PROJECT_PATH = "AutoSolve_ML_Data" #@param {type:"string"}

import os
import shutil

in_colab = False
try:
    import google.colab
    in_colab = True
except ImportError:
    pass

if in_colab and USE_GOOGLE_DRIVE:
    print("Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    
    persist_dir = f"/content/drive/MyDrive/{DRIVE_PROJECT_PATH}"
    os.makedirs(persist_dir, exist_ok=True)
    
    persistent_clips_dir = os.path.join(persist_dir, "clips")
    persistent_data_dir = os.path.join(persist_dir, "data")
    persistent_runs_dir = os.path.join(persist_dir, "runs")
    
    for d in [persistent_clips_dir, persistent_data_dir, persistent_runs_dir]:
        os.makedirs(d, exist_ok=True)
        
    print(f"\n📂 Google Drive paths mapped:")
    print(f"   Clips folder:    {persistent_clips_dir}")
    print(f"   Datasets folder: {persistent_data_dir}")
    print(f"   Runs folder:     {persistent_runs_dir}")
    
    # Link directories
    for path, target in [("ml/clips", persistent_clips_dir), ("ml/data", persistent_data_dir), ("ml/runs", persistent_runs_dir)]:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        if os.path.exists(path):
            if os.path.islink(path): os.unlink(path)
            elif os.path.isdir(path): shutil.rmtree(path)
            else: os.remove(path)
        os.symlink(target, path)
        print(f"✅ Linked '{path}' -> '{target}'")
else:
    print("Using temporary local filesystem inside notebook environment.")
    for d in ['ml/clips', 'ml/data', 'ml/runs', 'ml/data/raw', 'ml/data/processed', 'ml/data/live']:
        os.makedirs(d, exist_ok=True)
    print("👉 Upload your video clips directly to 'ml/clips/' and solve logs to 'ml/data/raw/'.")

In [ ]:
# Install dependencies
!pip install -q torch numpy onnx onnxruntime opencv-python matplotlib

import torch, numpy as np, onnx, onnxruntime as ort
print(f"PyTorch:     {torch.__version__}")
print(f"NumPy:       {np.__version__}")
print(f"ONNX:        {onnx.__version__}")
print(f"onnxruntime: {ort.__version__}")
print(f"GPU (CUDA):  {torch.cuda.is_available()}")

## 🛠️ Step 1: Standalone Schemas and Metadata Definitions

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import json
import matplotlib.pyplot as plt

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


## 🎥 Step 2: Feature Ingestion & Dataset Preparation

In [ ]:
import cv2

def extract_features_from_video(video_path: str) -> dict:
    """Extract average motion, zoom, distortion, and noise metrics directly from clip."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video file: {video_path}")

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    step = max(1, frame_count // 30)
    prev_gray = None
    motions, divergences, noises, curvatures = [], [], [], []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret: break

        if frame_idx % step == 0:
            small_frame = cv2.resize(frame, (320, 180))
            gray = cv2.cvtColor(small_frame, cv2.COLOR_BGR2GRAY)

            # Grain Noise
            blurred = cv2.GaussianBlur(gray, (5, 5), 0)
            high_freq = cv2.absdiff(gray, blurred)
            noises.append(float(np.mean(high_freq)))

            # Line Curvature
            edges = cv2.Canny(gray, 50, 150)
            lines = cv2.HoughLinesP(edges, 1, np.pi/180, 50, minLineLength=30, maxLineGap=10)
            if lines is not None:
                angles = [np.arctan2(l[0][3] - l[0][1], l[0][2] - l[0][0]) * 180 / np.pi for l in lines]
                curvatures.append(float(np.var(angles)) if angles else 0.0)
            else:
                curvatures.append(0.0)

            # Dense Flow
            if prev_gray is not None:
                flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
                fx, fy = flow[..., 0], flow[..., 1]
                motions.append(float(np.mean(np.sqrt(fx**2 + fy**2))))
                div = np.gradient(fx, axis=1) + np.gradient(fy, axis=0)
                divergences.append(float(np.mean(div)))
            prev_gray = gray
        frame_idx += 1

    cap.release()
    return {
        "clip_name": os.path.splitext(os.path.basename(video_path))[0],
        "width": width, "height": height, "fps": fps, "frame_count": frame_count,
        "mean_motion": float(np.mean(motions)) if motions else 0.0,
        "zoom_divergence": float(np.mean(np.abs(divergences))) if divergences else 0.0,
        "distortion_factor": float(np.mean(curvatures)) if curvatures else 0.0,
        "noise_ratio": float(np.mean(noises)) if noises else 0.0
    }

In [ ]:
import math
import random

def classify_region(fx: float, fy: float) -> str:
    ry = "top" if fy > 0.66 else ("bottom" if fy < 0.33 else "mid")
    rx = "left" if fx < 0.33 else ("right" if fx > 0.66 else "center")
    return "center" if (ry == "mid" and rx == "center") else (f"{ry}-{rx}" if ry != "mid" else f"mid-{rx}")

def run_opencv_tracking(video_path: str, grid_size: int = 8) -> Tuple[List[List[Tuple[float, float]]], dict]:
    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 24.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    ret, frame = cap.read()
    gray_prev = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    pts = cv2.goodFeaturesToTrack(gray_prev, maxCorners=grid_size*grid_size, qualityLevel=0.01, minDistance=20)
    if pts is None:
        pts = np.array([[[x, y]] for y in np.linspace(height*0.1, height*0.9, grid_size) for x in np.linspace(width*0.1, width*0.9, grid_size)], dtype=np.float32)

    num_tracks = len(pts)
    trajectories = [[(float(p[0][0]/width), float(p[0][1]/height))] for p in pts]
    active = np.ones(num_tracks, dtype=bool)
    p_prev = pts.copy()

    while True:
        ret, frame = cap.read()
        if not ret: break
        gray_curr = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        p_curr, status, _ = cv2.calcOpticalFlowPyrLK(gray_prev, gray_curr, p_prev, None, winSize=(21, 21), maxLevel=3)

        for idx in range(num_tracks):
            if not active[idx]: continue
            if status[idx][0] == 0:
                active[idx] = False
                continue
            x, y = p_curr[idx][0]
            if x < 0 or x >= width or y < 0 or y >= height:
                active[idx] = False
                continue
            trajectories[idx].append((float(x/width), float(y/height)))
        p_prev = p_curr
        gray_prev = gray_curr
    cap.release()
    return trajectories, {"clip_name": os.path.splitext(os.path.basename(video_path))[0], "width": width, "height": height, "fps": fps, "frame_count": frame_count}

In [ ]:
def simulate_variation(
    base_trajectories: List[List[Tuple[float, float]]],
    meta: Dict[str, Any],
    quality: str,
    robust: bool,
    tripod: bool
) -> Dict[str, Any]:
    """
    Simulates tracking settings variations on top of the base video trajectories.
    Injects synthetic noise, occlusions/failures, and computes velocities/jitter.
    """
    import random, math
    random.seed(hash(f"{meta['clip_name']}_{quality}_{robust}_{tripod}") % 1234567)

    # Adjust parameters based on tracking preset
    if quality == "QUALITY":
        noise_std = 0.001
        survival_base_prob = 0.85
    elif quality == "FAST":
        noise_std = 0.005
        survival_base_prob = 0.50
    else:  # BALANCED
        noise_std = 0.002
        survival_base_prob = 0.70

    if robust:
        survival_base_prob += 0.15
        noise_std *= 0.8

    survival_base_prob = min(0.95, max(0.20, survival_base_prob))
    track_samples = []
    survived_count = 0

    for idx_t, base_coords in enumerate(base_trajectories):
        if len(base_coords) < 6:
            continue

        fx, fy = base_coords[0]
        region = classify_region(fx, fy)

        # Region-based survival multiplier
        region_survival_mult = 1.0
        if "top" in region:
            region_survival_mult *= 0.7
        if "left" in region or "right" in region:
            region_survival_mult *= 0.85
        if "center" in region:
            region_survival_mult *= 1.1

        survival_prob = min(0.98, survival_base_prob * region_survival_mult)
        survived = random.random() < survival_prob
        has_bundle = survived and (random.random() < 0.90)

        coords = []
        if survived:
            for x, y in base_coords:
                nx = x + random.normalvariate(0.0, noise_std)
                ny = y + random.normalvariate(0.0, noise_std)
                coords.append((max(0.0, min(1.0, nx)), max(0.0, min(1.0, ny))))
            survived_count += 1
            average_error = random.uniform(0.15, 0.45)
        else:
            fail_frame = random.randint(5, len(base_coords) - 1)
            slip_frames = min(5, fail_frame)
            for f_idx in range(fail_frame):
                x, y = base_coords[f_idx]
                if f_idx >= fail_frame - slip_frames:
                    drift_factor = (f_idx - (fail_frame - slip_frames) + 1) * 0.01
                    nx = x + random.normalvariate(0.0, noise_std + drift_factor)
                    ny = y + random.normalvariate(0.0, noise_std + drift_factor)
                else:
                    nx = x + random.normalvariate(0.0, noise_std)
                    ny = y + random.normalvariate(0.0, noise_std)
                coords.append((max(0.0, min(1.0, nx)), max(0.0, min(1.0, ny))))
            average_error = random.uniform(0.8, 5.0)

        velocities = []
        for i in range(1, len(coords)):
            velocities.append((coords[i][0] - coords[i-1][0], coords[i][1] - coords[i-1][1]))

        jitter_scores = []
        for i in range(1, len(velocities)):
            dv_x = velocities[i][0] - velocities[i-1][0]
            dv_y = velocities[i][1] - velocities[i-1][1]
            jitter_scores.append(math.sqrt(dv_x**2 + dv_y**2))

        track_sample = {
            "track_name": f"track_{idx_t:03d}",
            "region": region,
            "positions": coords,
            "velocities": velocities,
            "jitter_scores": jitter_scores,
            "lifespan": len(coords),
            "survived": survived,
            "has_bundle": has_bundle,
            "average_error": average_error
        }
        track_samples.append(track_sample)

    total_tracks = len(track_samples)
    bundle_ratio = 0.0
    solve_success = False
    solve_error = 99.0

    if total_tracks > 0:
        bundle_count = sum(1 for t in track_samples if t["has_bundle"])
        bundle_ratio = bundle_count / total_tracks
        if bundle_count >= 8 and bundle_ratio >= 0.40:
            solve_success = True
            solve_error = max(0.1, random.normalvariate(0.4 + noise_std * 100, 0.15))
            if tripod:
                solve_error *= 0.8

    settings_dict = {
        "quality_preset": quality,
        "footage_type": "AUTO",
        "robust_mode": robust,
        "tripod_mode": tripod,
        "pattern_size": 11 if quality == "FAST" else (17 if quality == "BALANCED" else 31),
        "search_size": 51 if quality == "FAST" else (71 if quality == "BALANCED" else 121),
        "correlation": 0.55 if quality == "FAST" else (0.70 if quality == "BALANCED" else 0.85),
        "threshold": 0.40 if quality == "FAST" else (0.30 if quality == "BALANCED" else 0.15),
        "motion_model": "LocRot"
    }

    solve_sample = {
        "clip_metadata": meta,
        "settings": settings_dict,
        "tracks": track_samples,
        "solve_success": solve_success,
        "solve_error": solve_error,
        "bundle_count": sum(1 for t in track_samples if t["has_bundle"]),
        "bundle_ratio": bundle_ratio,
        "runtime_seconds": random.uniform(2.0, 15.0)
    }
    return solve_sample

def generate_synthetic_track(idx_t, survived):
    import random, math
    x, y = random.uniform(0.1, 0.9), random.uniform(0.1, 0.9)
    region = classify_region(x, y)
    lifespan = 100 if survived else random.randint(15, 95)
    positions = [(x, y)]
    step_std = 0.002
    for _ in range(1, lifespan):
        dx = random.normalvariate(0.0, step_std)
        dy = random.normalvariate(0.0, step_std)
        if not survived and _ >= lifespan - 5:
            dx += random.uniform(-0.015, 0.015)
            dy += random.uniform(-0.015, 0.015)
        x = max(0.0, min(1.0, x + dx))
        y = max(0.0, min(1.0, y + dy))
        positions.append((x, y))
        
    velocities = []
    for i in range(1, len(positions)):
        velocities.append((positions[i][0] - positions[i-1][0], positions[i][1] - positions[i-1][1]))
        
    jitter_scores = []
    for i in range(1, len(velocities)):
        dv_x = velocities[i][0] - velocities[i-1][0]
        dv_y = velocities[i][1] - velocities[i-1][1]
        jitter_scores.append(math.sqrt(dv_x**2 + dv_y**2))
        
    has_bundle = survived and (random.random() < 0.9)
    average_error = random.uniform(0.15, 0.45) if survived else random.uniform(0.8, 5.0)
    
    return {
        "track_name": f"t_{idx_t}",
        "region": region,
        "positions": positions,
        "velocities": velocities,
        "jitter_scores": jitter_scores,
        "lifespan": lifespan,
        "survived": survived,
        "has_bundle": has_bundle,
        "average_error": average_error
    }

def simulate_solve_attempts(clips_dir="ml/clips", out_dir="ml/data/raw"):
    os.makedirs(out_dir, exist_ok=True)
    supported = {'.mp4', '.mov', '.avi', '.mkv', '.ogg', '.webm'}
    files = [os.path.join(clips_dir, f) for f in os.listdir(clips_dir) if os.path.splitext(f)[1].lower() in supported] if os.path.exists(clips_dir) else []
    
    if not files:
        print("No clips found in 'ml/clips/'. Generating synthetic trajectories for training...")
        for i in range(5):
            clip_name = f"dummy_clip_{i}"
            with open(os.path.join(out_dir, f"{clip_name}_video_meta.json"), 'w') as fh:
                json.dump({"clip_name": clip_name, "width": 1920, "height": 1080, "fps": 30.0, "frame_count": 250, "mean_motion": 0.6, "zoom_divergence": 0.0, "distortion_factor": 0.0, "noise_ratio": 0.003}, fh, indent=4)
            
            dummy_tracks = []
            for j in range(40):
                survived = (j % 2 == 0)
                dummy_tracks.append(generate_synthetic_track(j, survived))
                
            solve = {
                "clip_metadata": {"clip_name": clip_name, "width": 1920, "height": 1080, "fps": 30.0, "frame_count": 250},
                "settings": {"quality_preset": "BALANCED", "footage_type": "AUTO", "robust_mode": False, "tripod_mode": False, "pattern_size": 17, "search_size": 71, "correlation": 0.7, "threshold": 0.3, "motion_model": "LocRot"},
                "tracks": dummy_tracks,
                "solve_success": True, "solve_error": 0.3, "bundle_count": 20, "bundle_ratio": 0.5, "runtime_seconds": 5.0
            }
            with open(os.path.join(out_dir, f"{clip_name}_balanced_standard.json"), 'w') as fh:
                json.dump(solve, fh, indent=4)
        return
        
    print(f"Processing {len(files)} clips...")
    for fp in files:
        clip_name = os.path.splitext(os.path.basename(fp))[0]
        try:
            v_feats = extract_features_from_video(fp)
            with open(os.path.join(out_dir, f"{clip_name}_video_meta.json"), 'w', encoding='utf-8') as fh:
                json.dump(v_feats, fh, indent=4)
            
            trajectories, meta = run_opencv_tracking(fp)
            variations = [
                ("BALANCED", False, False, "balanced_standard"),
                ("FAST", False, False, "fast_standard"),
                ("QUALITY", False, False, "quality_standard"),
                ("BALANCED", True, False, "balanced_robust"),
                ("BALANCED", False, True, "balanced_tripod")
            ]
            for q, r, t, suffix in variations:
                solve = simulate_variation(trajectories, meta, q, r, t)
                with open(os.path.join(out_dir, f"{clip_name}_{suffix}.json"), 'w') as fh:
                    json.dump(solve, fh, indent=4)
        except Exception as e:
            print(f"  Failed {clip_name}: {e}")

In [ ]:
import os
import json
import random
import math

def prepare_dataset(data_dir="ml/data/raw", output_json="ml/data/processed/settings_dataset.json"):
    video_meta_lookup = {}
    real_samples = []
    if os.path.exists(data_dir):
        for f in os.listdir(data_dir):
            fp = os.path.join(data_dir, f)
            if f.endswith('_video_meta.json'):
                try:
                    with open(fp) as fh:
                        m = json.load(fh)
                        video_meta_lookup[m["clip_name"]] = m
                except:
                    pass
            elif f.endswith('.json') and not f.endswith('settings_dataset.json') and not f.endswith('recommended_defaults.json') and not f.endswith('track_predictor.json'):
                try:
                    with open(fp) as fh:
                        s = json.load(fh)
                        real_samples.append(s)
                except:
                    pass
                
    by_clip = {}
    for s in real_samples:
        if "clip_metadata" in s:
            by_clip.setdefault(s["clip_metadata"]["clip_name"], []).append(s)
            
    clip_names = list(by_clip.keys())
    random.seed(42)
    train_raw, val_raw = [], []
    
    if len(clip_names) > 1:
        random.shuffle(clip_names)
        split_idx = max(1, int(len(clip_names) * 0.8))
        train_clips = set(clip_names[:split_idx])
        for c, samples in by_clip.items():
            if c in train_clips:
                train_raw.extend(samples)
            else:
                val_raw.extend(samples)
    else:
        if clip_names:
            all_samples = by_clip[clip_names[0]]
            random.shuffle(all_samples)
            split_idx = max(1, int(len(all_samples) * 0.8))
            train_raw = all_samples[:split_idx]
            val_raw = all_samples[split_idx:]
            
    if not train_raw:
        print("No samples available. Generating synthetic settings dataset...")
        # Fallback synthetic generation to allow notebook to run end-to-end
        train_X = [np.random.randn(28).tolist() for _ in range(100)]
        train_y = [random.random() for _ in range(100)]
        val_X = [np.random.randn(28).tolist() for _ in range(25)]
        val_y = [random.random() for _ in range(25)]
    else:
        train_X = [extract_sample_features(s, video_meta_lookup.get(s["clip_metadata"]["clip_name"])) for s in train_raw]
        train_y = [calculate_reward(s) for s in train_raw]
        val_X = [extract_sample_features(s, video_meta_lookup.get(s["clip_metadata"]["clip_name"])) for s in val_raw]
        val_y = [calculate_reward(s) for s in val_raw]
    
    num_features = len(train_X[0])
    means, stds = [0.0]*num_features, [1.0]*num_features
    for j in range(num_features):
        # Exclude tripod, robust, one-hot/binary columns from scaling
        # Indices: 4 (tripod), 5 (robust), 10-23 (one-hot columns)
        if j in (4, 5) or (10 <= j < 24):
            means[j] = 0.0
            stds[j] = 1.0
            continue
        col = [train_X[i][j] for i in range(len(train_X))]
        means[j] = sum(col) / len(col)
        variance = sum((x - means[j])**2 for x in col) / len(col)
        stds[j] = math.sqrt(variance) if variance > 1e-8 else 1.0
        
    def normalize(X):
        return [[(val - m)/s for val, m, s in zip(row, means, stds)] for row in X]
        
    dataset = {
        "train": {"X": normalize(train_X), "y": train_y},
        "val": {"X": normalize(val_X), "y": val_y},
        "input_mean": means, "input_std": stds
    }
    os.makedirs(os.path.dirname(output_json), exist_ok=True)
    with open(output_json, 'w') as fh:
        json.dump(dataset, fh, indent=4)
    print(f"Ingestion completed. Dataset saved to {output_json}")


In [ ]:
# Execute Ingestion pipeline
simulate_solve_attempts()
prepare_dataset()

## 🧠 Step 3: Train Expected Reward Optimizer MLP (28 Features)

In [ ]:
class SettingsMLP(nn.Module):
    """Expected reward MLP mapping 28 features to 1 expected reward score."""
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(28, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.network(x)

def train_settings_optimizer(epochs=50):
    set_seed(42)
    with open("ml/data/processed/settings_dataset.json") as f:
        dataset = json.load(f)
    tx = torch.tensor(dataset["train"]["X"], dtype=torch.float32)
    ty = torch.tensor(dataset["train"]["y"], dtype=torch.float32).unsqueeze(1)
    vx = torch.tensor(dataset["val"]["X"], dtype=torch.float32)
    vy = torch.tensor(dataset["val"]["y"], dtype=torch.float32).unsqueeze(1)
    
    loader = DataLoader(TensorDataset(tx, ty), batch_size=16, shuffle=True)
    model = SettingsMLP()
    criterion = nn.BCELoss()
    optimizer = optim.AdamW(model.parameters(), lr=0.005, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    
    best_val_loss = float('inf')
    best_weights = None
    
    train_losses = []
    val_losses = []
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for bx, by in loader:
            optimizer.zero_grad()
            # Data Augmentation: add low-magnitude Gaussian noise to continuous features
            if model.training:
                noise = torch.randn_like(bx) * 0.01
                for col_idx in range(bx.shape[1]):
                    if col_idx in (4, 5) or (10 <= col_idx < 24):
                        noise[:, col_idx] = 0.0
                bx_augmented = bx + noise
            else:
                bx_augmented = bx
            loss = criterion(model(bx_augmented), by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item() * bx.size(0)
        train_loss /= len(tx)
        train_losses.append(train_loss)
        scheduler.step()
        
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(vx), vy).item()
        val_losses.append(val_loss)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = {k: v.cpu().numpy().tolist() for k, v in model.state_dict().items()}
            
        if (epoch+1) % 10 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")
            
    meta = {"weights": best_weights, "input_mean": dataset["input_mean"], "input_std": dataset["input_std"], "best_val_loss": best_val_loss}
    os.makedirs("ml/runs/settings_optimizer", exist_ok=True)
    with open("ml/runs/settings_optimizer/model_meta_weights.json", 'w') as fh:
        json.dump(meta, fh, indent=4)
    print("Settings Optimizer weights saved successfully.")
    
    # Visualise training curve
    plt.figure(figsize=(8, 4))
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Settings Expected-Reward Model Training")
    plt.legend()
    plt.grid(True)
    plt.show()

train_settings_optimizer()


## 🎯 Step 4: Train Track Quality Predictor MLP (15 Features)

In [ ]:
class TrackMLP(nn.Module):
    """3-layer MLP for track survival prediction."""
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(15, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        return self.network(x)

def train_track_predictor(epochs=100):
    set_seed(42)
    # Gather data from raw solves
    data_dir = "ml/data/raw"
    json_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith('.json') and not f.endswith('_video_meta.json') and not f.endswith('settings_dataset.json') and not f.endswith('recommended_defaults.json') and not f.endswith('track_predictor.json')] if os.path.exists(data_dir) else []
    
    X_samples, y_samples = [], []
    clip_ids = []
    FOOTAGE_MAP = {f: i for i, f in enumerate(FOOTAGE_TYPES)}
    REGIONS_LIST = ['top-left', 'top-center', 'top-right', 'mid-left', 'center', 'mid-right', 'bottom-left', 'bottom-center', 'bottom-right']
    
    for fp in json_files:
        try:
            clip_name = os.path.splitext(os.path.basename(fp))[0]
            with open(fp) as fh:
                d = json.load(fh)
            f_idx = FOOTAGE_MAP.get(d["settings"]["footage_type"], 0)
            robust = d["settings"]["robust_mode"]
            tracks = d["tracks"]
            t_coords = {t["track_name"]: t["positions"] for t in tracks}
            for t in tracks:
                coords = t["positions"]
                survived = t["survived"] and t["has_bundle"]
                r_idx = REGIONS_LIST.index(t["region"]) if t["region"] in REGIONS_LIST else 4
                if len(coords) < 6:
                    continue
                for f_idx_check in range(5, len(coords), 10):
                    sub = coords[:f_idx_check+1]
                    neighbors = [tc[:f_idx_check+1] for name, tc in t_coords.items() if name != t["track_name"] and len(tc) > f_idx_check]
                    # Extracted features
                    feats = np.zeros(15, dtype=np.float32)
                    inspect = sub[-6:]
                    v_x = [inspect[i][0] - inspect[i-1][0] for i in range(1, len(inspect)) ]
                    v_y = [inspect[i][1] - inspect[i-1][1] for i in range(1, len(inspect)) ]
                    feats[0], feats[1] = np.mean(v_x), np.mean(v_y)
                    feats[2] = np.std(v_x) if len(v_x) > 1 else 0.0
                    feats[3] = np.std(v_y) if len(v_y) > 1 else 0.0
                    acc_x = [v_x[i] - v_x[i-1] for i in range(1, len(v_x))] if len(v_x) > 1 else [0.0]
                    acc_y = [v_y[i] - v_y[i-1] for i in range(1, len(v_y))] if len(v_y) > 1 else [0.0]
                    feats[4], feats[5] = np.mean(acc_x), np.mean(acc_y)
                    feats[6] = sum(1 for i in range(1, len(v_x)) if (v_x[i] > 0) != (v_x[i-1] > 0))
                    feats[7] = sum(1 for i in range(1, len(v_y)) if (v_y[i] > 0) != (v_y[i-1] > 0))
                    feats[8] = len(coords)
                    # Neighbor features
                    min_dist = 999.0
                    n_vels_x, n_vels_y = [], []
                    for n_coords in neighbors:
                        if len(n_coords) >= 1:
                            dist = ((coords[-1][0] - n_coords[-1][0])**2 + (coords[-1][1] - n_coords[-1][1])**2)**0.5
                            if dist < min_dist:
                                min_dist = dist
                            if len(n_coords) >= 2:
                                n_vels_x.append(n_coords[-1][0] - n_coords[-2][0])
                                n_vels_y.append(n_coords[-1][1] - n_coords[-2][1])
                    feats[9] = min_dist if min_dist < 998.0 else 1.0
                    if n_vels_x:
                        feats[10] = v_x[-1] - np.mean(n_vels_x)
                        feats[11] = v_y[-1] - np.mean(n_vels_y)
                    feats[12], feats[13] = float(r_idx), float(f_idx)
                    feats[14] = 1.0 if robust else 0.0
                    
                    X_samples.append(feats)
                    y_samples.append(1.0 if survived else 0.0)
                    clip_ids.append(clip_name)
        except Exception as e:
            print(f"Skipped {fp}: {e}")
        
    if not X_samples:
        print("No track samples found. Injecting synthetic vectors for training...")
        # Since we have no real data, let's generate balanced synthetic data using random walk
        for i in range(5):
            clip_name = f"dummy_clip_{i}"
            for j in range(40):
                survived = (j % 2 == 0)
                t = generate_synthetic_track(j, survived)
                coords = t["positions"]
                r_idx = REGIONS_LIST.index(t["region"]) if t["region"] in REGIONS_LIST else 4
                if len(coords) < 6:
                    continue
                for f_idx_check in range(5, len(coords), 10):
                    sub = coords[:f_idx_check+1]
                    feats = np.zeros(15, dtype=np.float32)
                    inspect = sub[-6:]
                    v_x = [inspect[i][0] - inspect[i-1][0] for i in range(1, len(inspect))]
                    v_y = [inspect[i][1] - inspect[i-1][1] for i in range(1, len(inspect))]
                    feats[0], feats[1] = np.mean(v_x), np.mean(v_y)
                    feats[2] = np.std(v_x) if len(v_x) > 1 else 0.0
                    feats[3] = np.std(v_y) if len(v_y) > 1 else 0.0
                    acc_x = [v_x[i] - v_x[i-1] for i in range(1, len(v_x))] if len(v_x) > 1 else [0.0]
                    acc_y = [v_y[i] - v_y[i-1] for i in range(1, len(v_y))] if len(v_y) > 1 else [0.0]
                    feats[4], feats[5] = np.mean(acc_x), np.mean(acc_y)
                    feats[6] = sum(1 for i in range(1, len(v_x)) if (v_x[i] > 0) != (v_x[i-1] > 0))
                    feats[7] = sum(1 for i in range(1, len(v_y)) if (v_y[i] > 0) != (v_y[i-1] > 0))
                    feats[8] = len(coords)
                    feats[9] = 0.1
                    feats[10] = 0.0
                    feats[11] = 0.0
                    feats[12], feats[13] = float(r_idx), 0.0
                    feats[14] = 0.0
                    X_samples.append(feats)
                    y_samples.append(1.0 if survived else 0.0)
                    clip_ids.append(clip_name)
        
    X_arr = np.array(X_samples, dtype=np.float32)
    y_arr = np.array(y_samples, dtype=np.float32)
    
    # Address class imbalance
    pos_ratio = sum(y_samples) / len(y_samples) if y_samples else 0.5
    print(f"Class balance: {pos_ratio:.1%} survived")
    pos_weight = torch.tensor([(1.0 - pos_ratio) / max(1e-5, pos_ratio)], dtype=torch.float32)
    
    # Grouped split by clip
    unique_clips = list(set(clip_ids))
    random.seed(42)
    random.shuffle(unique_clips)
    split_idx = max(1, int(len(unique_clips) * 0.8))
    train_clips = set(unique_clips[:split_idx])
    
    train_indices = [i for i, c in enumerate(clip_ids) if c in train_clips]
    val_indices = [i for i, c in enumerate(clip_ids) if c not in train_clips]
    
    X_train_raw = X_arr[train_indices]
    y_train = y_arr[train_indices]
    X_val_raw = X_arr[val_indices]
    y_val = y_arr[val_indices]
    
    # Normalization parameters from training set only
    mean = np.mean(X_train_raw, axis=0) if len(X_train_raw) > 0 else np.zeros(15)
    std = np.std(X_train_raw, axis=0) if len(X_train_raw) > 0 else np.ones(15)
    std[std < 1e-6] = 1.0
    
    X_train_norm = (X_train_raw - mean) / std if len(X_train_raw) > 0 else X_train_raw
    X_val_norm = (X_val_raw - mean) / std if len(X_val_raw) > 0 else X_val_raw
    
    tx, ty = torch.tensor(X_train_norm), torch.tensor(y_train).unsqueeze(1)
    vx, vy = torch.tensor(X_val_norm), torch.tensor(y_val).unsqueeze(1)
    
    loader = DataLoader(TensorDataset(tx, ty), batch_size=32, shuffle=True)
    model = TrackMLP()
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    
    best_val_loss = float('inf')
    best_weights = None
    
    train_losses = []
    val_losses = []
    val_accuracies = []
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for bx, by in loader:
            optimizer.zero_grad()
            # Data Augmentation: add low-magnitude Gaussian noise to continuous features
            if model.training:
                noise = torch.randn_like(bx) * 0.01
                for col_idx in range(bx.shape[1]):
                    if col_idx in (12, 13, 14):
                        noise[:, col_idx] = 0.0
                bx_augmented = bx + noise
            else:
                bx_augmented = bx
            loss = criterion(model(bx_augmented), by)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()*bx.size(0)
        train_loss /= len(tx)
        train_losses.append(train_loss)
        scheduler.step()
        
        model.eval()
        with torch.no_grad():
            logits = model(vx)
            val_loss = criterion(logits, vy).item()
            probs = torch.sigmoid(logits)
            acc = np.mean((probs.numpy() > 0.5) == vy.numpy()) if len(vy) > 0 else 1.0
        val_losses.append(val_loss)
        val_accuracies.append(acc)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = {k: v.cpu().numpy().tolist() for k, v in model.state_dict().items()}
        if (epoch+1)%20 == 0 or epoch == 0:
            print(f"Epoch {epoch+1:03d}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {acc:.1%}")
            
    meta = {"weights": best_weights, "input_mean": mean.tolist(), "input_std": std.tolist()}
    os.makedirs("ml/runs/track_predictor", exist_ok=True)
    with open("ml/runs/track_predictor/model_meta_weights.json", 'w') as fh:
        json.dump(meta, fh, indent=4)
    print("Track Predictor saved successfully.")
    
    # Plot training curves
    fig, ax1 = plt.subplots(figsize=(8, 4))
    ax1.plot(train_losses, label="Train Loss", color="blue")
    ax1.plot(val_losses, label="Val Loss", color="red")
    ax1.set_ylabel("Loss")
    ax1.legend(loc="upper left")
    
    ax2 = ax1.twinx()
    ax2.plot(val_accuracies, label="Val Acc", color="green", linestyle="--")
    ax2.set_ylabel("Accuracy")
    ax2.legend(loc="upper right")
    plt.title("Track Predictor Training Curves")
    plt.show()

train_track_predictor()


## 📈 Step 5: Empirical Trackability Heatmap / Region Weights

In [ ]:
import os
assert os.path.exists("ml/data/raw") and len([f for f in os.listdir("ml/data/raw") if f.endswith('.json')]) > 0, "Run Step 2 first!"

def compile_region_weights(data_dir="ml/data/raw", output_json="ml/runs/region_weights.json"):
    regions = ['top-left', 'top-center', 'top-right', 'mid-left', 'center', 'mid-right', 'bottom-left', 'bottom-center', 'bottom-right']
    prebaked = {
        "AUTO": {r: 1.0 for r in regions},
        "INDOOR": {r: 1.0 for r in regions},
        "OUTDOOR": {"top-left": 0.40, "top-center": 0.15, "top-right": 0.40, "mid-left": 0.85, "center": 0.95, "mid-right": 0.85, "bottom-left": 1.00, "bottom-center": 1.00, "bottom-right": 1.00},
        "DRONE": {"top-left": 0.10, "top-center": 0.05, "top-right": 0.10, "mid-left": 0.70, "center": 0.90, "mid-right": 0.70, "bottom-left": 1.00, "bottom-center": 1.00, "bottom-right": 1.00}
    }
    json_files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith('.json') and not f.endswith('_video_meta.json') and not f.endswith('settings_dataset.json')] if os.path.exists(data_dir) else []
    weights = {}
    if json_files:
        stats = {}
        for fp in json_files:
            try:
                with open(fp) as fh: d = json.load(fh)
                f_type = d["settings"]["footage_type"]
                stats.setdefault(f_type, {})
                for t in d["tracks"]:
                    region = t["region"]
                    r_stats = stats[f_type].setdefault(region, {"detected": 0, "survived": 0})
                    r_stats["detected"] += 1
                    if t["survived"] and t["has_bundle"]: r_stats["survived"] += 1
            except: pass
            
        for f_type in ["AUTO", "INDOOR", "OUTDOOR", "DRONE"]:
            weights[f_type] = {}
            f_stats = stats.get(f_type, {})
            fallback_weights = prebaked.get(f_type, prebaked["AUTO"])
            for region, r_weights in fallback_weights.items():
                if region in f_stats and f_stats[region]["detected"] > 10:
                    weights[f_type][region] = f_stats[region]["survived"] / f_stats[region]["detected"]
                else:
                    weights[f_type][region] = r_weights
            max_w = max(weights[f_type].values())
            if max_w > 0:
                for r in weights[f_type]: weights[f_type][r] = round(weights[f_type][r]/max_w, 3)
    else:
        weights = prebaked
        
    with open(output_json, 'w') as fh:
        json.dump(weights, fh, indent=4)
    print(f"Empirical weights generated under {output_json}")

compile_region_weights()

## 📤 Step 6: Standalone Exporters (ONNX, NumPy, & Defaults Grid Search)

In [ ]:
def export_onnx_binaries(out_dir="autosolve/tracker/models"):
    import json
    import os
    import numpy as np
    import torch
    import torch.nn as nn
    
    os.makedirs(out_dir, exist_ok=True)
    
    # 1. Settings Optimizer model
    with open("ml/runs/settings_optimizer/model_meta_weights.json") as fh:
        s_meta = json.load(fh)
    s_model = SettingsMLP()
    s_model.load_state_dict({k: torch.tensor(v) for k, v in s_meta["weights"].items()})
    s_model.eval()
    
    s_onnx = os.path.join(out_dir, "settings_model.onnx")
    torch.onnx.export(
        s_model, torch.zeros(1, 28), s_onnx,
        input_names=["clip_features"], output_names=["reward"], opset_version=17,
        dynamic_axes={"clip_features": {0: "batch"}, "reward": {0: "batch"}}
    )
    with open(os.path.join(out_dir, "settings_model_meta.json"), 'w') as fh:
        json.dump({"input_mean": s_meta["input_mean"], "input_std": s_meta["input_std"], "input_size": 28, "output_size": 1}, fh, indent=2)
        
    # 2. Track Predictor model (15 Features) - wrap with Sigmoid for ONNX
    with open("ml/runs/track_predictor/model_meta_weights.json") as fh:
        t_meta = json.load(fh)
    t_model = TrackMLP()
    t_model.load_state_dict({k: torch.tensor(v) for k, v in t_meta["weights"].items()})
    t_model.eval()
    
    class TrackONNXWrapper(nn.Module):
        def __init__(self, base_model):
            super().__init__()
            self.base_model = base_model
        def forward(self, x):
            return torch.sigmoid(self.base_model(x))
            
    wrapped_t_model = TrackONNXWrapper(t_model)
    wrapped_t_model.eval()
    
    t_onnx = os.path.join(out_dir, "track_predictor.onnx")
    torch.onnx.export(
        wrapped_t_model, torch.zeros(1, 15), t_onnx,
        input_names=["track_features"], output_names=["survival_prob"], opset_version=17,
        dynamic_axes={"track_features": {0: "batch"}, "survival_prob": {0: "batch"}}
    )
    with open(os.path.join(out_dir, "track_predictor_meta.json"), 'w') as fh:
        json.dump({"input_mean": t_meta["input_mean"], "input_std": t_meta["input_std"], "input_size": 15, "output_size": 1}, fh, indent=2)
        
    # 3. NumPy JSON model with BatchNorm folding
    weights = t_meta["weights"]
    
    def fold_bn(linear_w, linear_b, bn_w, bn_b, bn_mean, bn_var, eps=1e-5):
        lw = np.array(linear_w, dtype=np.float32)
        lb = np.array(linear_b, dtype=np.float32)
        bw = np.array(bn_w, dtype=np.float32)
        bb = np.array(bn_b, dtype=np.float32)
        bm = np.array(bn_mean, dtype=np.float32)
        bv = np.array(bn_var, dtype=np.float32)
        
        scale = bw / np.sqrt(bv + eps)
        w_fold = lw * scale[:, np.newaxis]
        b_fold = (lb - bm) * scale + bb
        return w_fold.tolist(), b_fold.tolist()

    print("Folding BatchNorm layers for NumPy JSON model...")
    l1_w, l1_b = fold_bn(
        weights["network.0.weight"], weights["network.0.bias"],
        weights["network.1.weight"], weights["network.1.bias"],
        weights["network.1.running_mean"], weights["network.1.running_var"]
    )
    l2_w, l2_b = fold_bn(
        weights["network.4.weight"], weights["network.4.bias"],
        weights["network.5.weight"], weights["network.5.bias"],
        weights["network.5.running_mean"], weights["network.5.running_var"]
    )
    l3_w = weights["network.8.weight"]
    l3_b = weights["network.8.bias"]

    numpy_model = {
        "layer1_weight": l1_w,
        "layer1_bias": l1_b,
        "layer2_weight": l2_w,
        "layer2_bias": l2_b,
        "layer3_weight": l3_w,
        "layer3_bias": l3_b,
        "input_mean": t_meta["input_mean"],
        "input_std": t_meta["input_std"],
        "activation": "relu"
    }
    
    # Also save inside runs and in model folder
    with open("ml/runs/track_predictor.json", 'w') as fh:
        json.dump(numpy_model, fh, indent=4)
    with open(os.path.join(out_dir, "track_predictor.json"), 'w') as fh:
        json.dump(numpy_model, fh, indent=4)
        
    print("ONNX and JSON models exported successfully!")

export_onnx_binaries()


In [ ]:
def predict_rewards(X, weights):
    import numpy as np
    x = np.array(X, dtype=np.float32)
    
    # Fold BN for settings model
    def fold_bn(lw, lb, bn_w, bn_b, bn_mean, bn_var, eps=1e-5):
        scale = bn_w / np.sqrt(bn_var + eps)
        w_fold = lw * scale[:, np.newaxis]
        b_fold = (lb - bn_mean) * scale + bn_b
        return w_fold, b_fold

    if "network.1.weight" in weights:
        # Fold first layer
        w0, b0 = fold_bn(
            np.array(weights["network.0.weight"], dtype=np.float32),
            np.array(weights["network.0.bias"], dtype=np.float32),
            np.array(weights["network.1.weight"], dtype=np.float32),
            np.array(weights["network.1.bias"], dtype=np.float32),
            np.array(weights["network.1.running_mean"], dtype=np.float32),
            np.array(weights["network.1.running_var"], dtype=np.float32)
        )
        # Fold second layer
        w2, b2 = fold_bn(
            np.array(weights["network.4.weight"], dtype=np.float32),
            np.array(weights["network.4.bias"], dtype=np.float32),
            np.array(weights["network.5.weight"], dtype=np.float32),
            np.array(weights["network.5.bias"], dtype=np.float32),
            np.array(weights["network.5.running_mean"], dtype=np.float32),
            np.array(weights["network.5.running_var"], dtype=np.float32)
        )
        w4 = np.array(weights["network.8.weight"], dtype=np.float32)
        b4 = np.array(weights["network.8.bias"], dtype=np.float32)
    else:
        # Legacy/old architecture fallback
        w0 = np.array(weights["network.0.weight"], dtype=np.float32)
        b0 = np.array(weights["network.0.bias"], dtype=np.float32)
        w2 = np.array(weights["network.2.weight"], dtype=np.float32)
        b2 = np.array(weights["network.2.bias"], dtype=np.float32)
        w4 = np.array(weights["network.4.weight"], dtype=np.float32)
        b4 = np.array(weights["network.4.bias"], dtype=np.float32)
        
    x1 = np.maximum(0.0, x @ w0.T + b0)
    x2 = np.maximum(0.0, x1 @ w2.T + b2)
    logits = x2 @ w4.T + b4
    x3 = 1.0 / (1.0 + np.exp(-np.clip(logits, -20.0, 20.0)))
    return x3.flatten().tolist()


In [ ]:
# Outputs Verification
expected = [
    'ml/runs/onnx/track_predictor.onnx',
    'ml/runs/onnx/track_predictor_meta.json',
    'ml/runs/onnx/settings_model.onnx',
    'ml/runs/onnx/settings_model_meta.json',
    'ml/runs/track_predictor.json',
    'ml/runs/defaults.json',
    'ml/runs/region_weights.json',
    'ml/runs/recommended_defaults.json',
]
print("Trained Model Output Verification:\n")
for f in expected:
    exists = os.path.exists(f)
    size   = os.path.getsize(f) // 1024 if exists else 0
    status = f'✅  {size:4d} KB' if exists else '❌  MISSING'
    print(f'{status}   {f}')


In [ ]:
# Colab direct download handler
try:
    from google.colab import files
    print("Downloading files directly to browser:")
    for f in expected:
        if os.path.exists(f):
            print(f'  Downloading: {f}')
            files.download(f)
except ImportError:
    print("Running locally. Models are located in your local project 'ml/runs/' folder.")

In [ ]:
#@title Copy Trained Models to Addon Directories (Local Environment Only)
COPY_TO_ADDON = True #@param {type:"boolean"}

if not in_colab and COPY_TO_ADDON:
    import shutil
    
    dest_models_dir = "autosolve/tracker/models"
    dest_presets_dir = "autosolve/tracker/presets"
    
    os.makedirs(dest_models_dir, exist_ok=True)
    os.makedirs(dest_presets_dir, exist_ok=True)
    
    copies = [
        ("ml/runs/onnx/settings_model.onnx", os.path.join(dest_models_dir, "settings_model.onnx")),
        ("ml/runs/onnx/settings_model_meta.json", os.path.join(dest_models_dir, "settings_model_meta.json")),
        ("ml/runs/onnx/track_predictor.onnx", os.path.join(dest_models_dir, "track_predictor.onnx")),
        ("ml/runs/onnx/track_predictor_meta.json", os.path.join(dest_models_dir, "track_predictor_meta.json")),
        ("ml/runs/track_predictor.json", os.path.join(dest_models_dir, "track_predictor.json")),
        ("ml/runs/defaults.json", os.path.join(dest_presets_dir, "defaults.json"))
    ]
    
    print("Copying trained models to Blender addon folders...")
    for src, dst in copies:
        if os.path.exists(src):
            shutil.copy2(src, dst)
            print(f"  ✓ Copied {src} -> {dst}")
        else:
            print(f"  ✗ Source not found: {src}")
else:
    print("Running in Colab or copy disabled. Download the files manually.")